# UGUIDU Museum RAG Evaluation Pipeline

Evaluates the UGUIDU voice assistant RAG pipeline for Stedelijk Museum Breda.

Research basis: RAGTruth (ACL 2024), RAGAS v0.2, Barnett 2024 Seven Failure Points,
Agresti-Coull Wilson intervals, Krippendorff (1980), Benjamini & Hochberg (1995).

**Pipeline overview:**
1. Load ground truth from PostgreSQL `knowledge_base` (31 BR% artifacts)
2. Build test cases from `LABEL_TO_DB` CV classifier labels
3. Query UGUIDU streaming API (`/api/chat`) for each label
4. GPT-5 judge — 3-run self-consistency with majority vote, 5 bias mitigations
5. Statistical analysis — Wilson CI, Krippendorff α, BH-FDR, phi-coefficient
6. Export `eval_results.csv` + `eval_summary.xlsx` (4 sheets)

**Resume:** checkpoint at `eval_output/eval_checkpoint.json` — re-run Cell 8 to continue.

## Cell 1 — Install Dependencies

In [ ]:
import subprocess
subprocess.run([
    'pip', 'install', '--quiet',
    'requests', 'pandas', 'tqdm', 'openpyxl',
    'scipy', 'statsmodels', 'krippendorff',
    'numpy', 'openai', 'psycopg2-binary'
], check=True)
print('All dependencies installed')

## Cell 2 — Configuration

Set `OPENAI_API_KEY` in your environment before running. Everything else is pre-configured.

In [ ]:
import os, re, json, time, uuid, logging, random
from pathlib import Path
from datetime import datetime
from collections import Counter

import requests
import pandas as pd
import numpy as np
from tqdm import tqdm
from openai import OpenAI

# ── Judge ────────────────────────────────────────────────────
OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY', '')
JUDGE_MODEL     = 'gpt-5'
JUDGE_RUNS      = 3

# ── UGUIDU backend ───────────────────────────────────────────
UGUIDU_API_URL  = 'http://194.171.191.226:3414/api/chat'
DEFAULT_AUDIENCE = 'tourist'
API_DELAY       = 1.5   # seconds between API calls

# ── PostgreSQL knowledge base ─────────────────────────────────
DB_HOST         = '194.171.191.227'
DB_PORT         = 5432
DB_USER         = 'postgres'
DB_PASSWORD     = 'postgres'
DB_NAME         = 'o2'

# ── CV classifier label → DB search term ─────────────────────
# Mirrors LABEL_TO_DB in llm_service.py exactly.
LABEL_TO_DB = {
    'Justinus of Nassau': 'Justinus van Nassau',
    'Frederick Henry of Orange-Nassau': 'Frederik Hendrik',
    'Maurice of Oranje-Nassau': 'Maurits',
    'Equestrian statue of William of Orange': 'ruiterstandbeeld Willem van Oranje',
    'Money of necessity minted during the siege of breda': 'Noodmunten',
    'City map of Breda in relief': 'Stadsplan in relief',
    'Model of a 24-pound gun': 'kanon',
    'Constantijn Huygens, curator van de Illustere School in Breda': 'Constantijn Huygens',
    'Beeld vna de turfschipper': 'turfschipper',
    'Departure of the Spanish occupation from Breda on 10 October 1637': 'Uittocht van de Spaanse bezetting',
    'Military Operations in flanders with siege and capture of breda': 'beleg van Breda',
    'Retreat of the Spanish Garrison from Breda': 'Uittocht van de Spaanse bezetting',
    'Obsidio Bredana': 'Obsidio Bredana',
    'het beleg van Breda': 'beleg van Breda',
}

# ── Outputs ───────────────────────────────────────────────────
OUTPUT_DIR      = Path('eval_output')
CKPT_PATH       = OUTPUT_DIR / 'eval_checkpoint.json'
OUTPUT_DIR.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    handlers=[
        logging.FileHandler(OUTPUT_DIR / 'eval.log'),
        logging.StreamHandler()
    ]
)
log = logging.getLogger('uguidu-eval')

openai_client = OpenAI(api_key=OPENAI_API_KEY)

print(f'Config loaded | Judge={JUDGE_MODEL} | Runs={JUDGE_RUNS} | Labels={len(LABEL_TO_DB)}')

## Cell 3 — Test Connections

In [ ]:
import psycopg2

def gpt_chat(messages, temperature=0.5, max_completion_tokens=900):
    """Call GPT-5 with exponential-backoff retry."""
    for attempt in range(4):
        try:
            resp = openai_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=messages,
                temperature=temperature,
                max_completion_tokens=max_completion_tokens
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            wait = 2 ** attempt
            log.warning(f'OpenAI attempt {attempt+1} failed: {e} — retrying in {wait}s')
            time.sleep(wait)
    raise RuntimeError('OpenAI failed after 4 attempts')

def get_db_conn():
    return psycopg2.connect(
        host=DB_HOST, port=DB_PORT,
        dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD,
        connect_timeout=10
    )

# Test OpenAI
print('Testing GPT-5...')
r = gpt_chat([{'role': 'user', 'content': 'Reply with the single word OK.'}],
             temperature=0.0, max_completion_tokens=10)
print(f'  {r}')

# Test PostgreSQL
print('\nTesting PostgreSQL...')
conn = get_db_conn()
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM knowledge_base WHERE artifact_id LIKE 'BR%'")
    n_artifacts = cur.fetchone()[0]
print(f'  BR% artifact records: {n_artifacts}')
conn.close()

# Test UGUIDU API
print('\nTesting UGUIDU API...')
test_payload = {
    'message': 'Hello',
    'audience': DEFAULT_AUDIENCE,
    'conversation_id': str(uuid.uuid4())
}
with requests.post(UGUIDU_API_URL, json=test_payload, stream=True, timeout=30) as resp:
    resp.raise_for_status()
    parts = []
    for line in resp.iter_lines():
        if line:
            raw = line.decode('utf-8') if isinstance(line, bytes) else line
            try:
                d = json.loads(raw)
                if d.get('type') == 'answer':
                    parts.append(d.get('content', ''))
            except json.JSONDecodeError:
                pass
    api_test_response = ''.join(parts).strip()
print(f'  Response: {api_test_response[:120]}')
print('\nAll connections OK')

## Cell 4 — Load Ground Truth from knowledge_base

Introspects the schema first, then loads all 31 BR% artifacts. Ground truth is
extracted deterministically from the DB — never by LLM.

In [ ]:
def load_all_artifacts():
    """Load all BR% records from knowledge_base."""
    conn = get_db_conn()
    try:
        with conn.cursor() as cur:
            # Introspect schema
            cur.execute("""
                SELECT column_name, data_type
                FROM information_schema.columns
                WHERE table_name = 'knowledge_base'
                ORDER BY ordinal_position
            """)
            schema_rows = cur.fetchall()
            print('knowledge_base schema:')
            for col, dtype in schema_rows:
                print(f'  {col}: {dtype}')

            # Load all BR% records
            cur.execute(
                "SELECT * FROM knowledge_base WHERE artifact_id LIKE 'BR%' ORDER BY artifact_id"
            )
            rows = cur.fetchall()
            col_names = [d[0] for d in cur.description]

        artifacts = []
        for row in rows:
            record = {k: (str(v) if v is not None else '') for k, v in zip(col_names, row)}
            artifacts.append(record)

        print(f'\nLoaded {len(artifacts)} BR% artifacts')
        return artifacts, col_names
    finally:
        conn.close()


def extract_content(artifact: dict, col_names: list) -> str:
    """Extract primary text content from an artifact record."""
    priority = ['content', 'description', 'text', 'body', 'information',
                'tekst', 'beschrijving', 'info', 'details']
    for col in priority:
        if col in artifact and len(artifact[col].strip()) > 30:
            return artifact[col].strip()
    # Fallback: concatenate all long non-meta columns
    skip = {'artifact_id', 'id', 'created_at', 'updated_at', 'modified_at'}
    parts = [v for k, v in artifact.items()
             if k not in skip and len(v.strip()) > 30]
    return '\n'.join(parts)


def extract_name(artifact: dict) -> str:
    """Extract primary display name from an artifact record."""
    for col in ['name', 'title', 'naam', 'titel', 'artifact_name', 'label', 'subject']:
        if col in artifact and artifact[col].strip():
            return artifact[col].strip()
    return artifact.get('artifact_id', 'Unknown')


all_artifacts, col_names = load_all_artifacts()

# Show sample record
if all_artifacts:
    print('\nSample artifact:')
    for k, v in all_artifacts[0].items():
        print(f'  {k}: {v[:100]}')

## Cell 5 — Build Test Cases

Each CV classifier label in `LABEL_TO_DB` becomes one test case. The DB search term
is matched against artifact names and content to retrieve the ground truth.

In [ ]:
def match_artifact(db_term: str, artifacts: list) -> dict | None:
    """Find the artifact whose name/content best matches a DB search term."""
    term_lower = db_term.lower()
    term_words = set(term_lower.split())

    # Pass 1: exact name match
    for a in artifacts:
        if extract_name(a).lower() == term_lower:
            return a

    # Pass 2: name contains the term
    for a in artifacts:
        if term_lower in extract_name(a).lower():
            return a

    # Pass 3: content contains the term
    for a in artifacts:
        content = extract_content(a, col_names).lower()
        if term_lower in content:
            return a

    # Pass 4: word-overlap score
    best_score, best_match = 0.0, None
    for a in artifacts:
        combined = (extract_name(a) + ' ' + extract_content(a, col_names)).lower()
        combined_words = set(combined.split())
        overlap = len(term_words & combined_words) / max(len(term_words), 1)
        if overlap > best_score:
            best_score, best_match = overlap, a
    return best_match if best_score >= 0.4 else None


test_cases = []
for cv_label, db_term in LABEL_TO_DB.items():
    matched = match_artifact(db_term, all_artifacts)
    if matched:
        gt = extract_content(matched, col_names)
        artifact_name = extract_name(matched)
        artifact_id = matched.get('artifact_id', '')
    else:
        gt, artifact_name, artifact_id = '', db_term, ''
        log.warning(f'No DB match for: {db_term!r}')

    test_cases.append({
        'id': f'TC_{len(test_cases)+1:02d}',
        'cv_label': cv_label,
        'db_search_term': db_term,
        'artifact_id': artifact_id,
        'artifact_name': artifact_name,
        'ground_truth': gt,
    })

print(f'Built {len(test_cases)} test cases:\n')
print(f'{"ID":<8} {"CV Label":<52} {"Artifact":<32} {"GT chars"}')
print('-' * 110)
for tc in test_cases:
    status = 'OK' if tc['ground_truth'] else 'NO MATCH'
    print(f'{tc["id"]:<8} {tc["cv_label"][:50]:<52} '
          f'{tc["artifact_name"][:30]:<32} {len(tc["ground_truth"]):>6}  [{status}]')

## Cell 6 — UGUIDU API Client

Sends the CV label as both `message` and `landmark` to trigger the `LABEL_TO_DB`
mapping path in `server.py`. Parses NDJSON streaming response.

In [ ]:
def query_uguidu(label: str, audience: str = DEFAULT_AUDIENCE) -> str:
    """Send CV label to UGUIDU and return the assembled streaming response."""
    payload = {
        'message': label,
        'landmark': label,          # triggers LABEL_TO_DB in server.py
        'audience': audience,
        'conversation_id': str(uuid.uuid4()),  # fresh ID per request
    }

    for attempt in range(4):
        parts = []
        try:
            with requests.post(
                UGUIDU_API_URL,
                json=payload,
                stream=True,
                timeout=90
            ) as resp:
                resp.raise_for_status()
                for raw_line in resp.iter_lines():
                    if not raw_line:
                        continue
                    line = raw_line.decode('utf-8') if isinstance(raw_line, bytes) else raw_line
                    try:
                        data = json.loads(line)
                        if data.get('type') == 'answer':
                            parts.append(data.get('content', ''))
                    except json.JSONDecodeError:
                        pass
            return ''.join(parts).strip()
        except Exception as e:
            wait = 2 ** attempt
            log.warning(f'UGUIDU attempt {attempt+1} failed: {e} — retrying in {wait}s')
            time.sleep(wait)

    return 'ERROR: UGUIDU API failed after 4 attempts'


# Smoke test with first test case
print(f'Smoke test: {test_cases[0]["cv_label"]}')
smoke_response = query_uguidu(test_cases[0]['cv_label'])
print(f'Response: {smoke_response[:200]}')
print('UGUIDU API client ready')

## Cell 7 — Self-Consistency Evaluator (3-Run Majority Vote)

Five bias mitigations from published literature:
1. **Position randomization** — GROUND_TRUTH and UGUIDU_RESPONSE order shuffled per run
2. **Temperature 0.5** — controlled stochasticity across runs
3. **Score-before-reasoning** — scores output first to prevent reasoning drift
4. **Majority vote** — binary dims: ≥2/3 runs; factual_accuracy: median; ragtruth_label: plurality
5. **Seeded shuffle** — deterministic per (artifact_id, run_idx) for reproducibility

Individual run binary scores are stored in `_run_binary` for Krippendorff alpha computation.

In [ ]:
JUDGE_SYSTEM = """You are a strict, impartial evaluator for a museum voice assistant RAG pipeline.

CORE RULES:
A. GROUND_TRUTH is the only source of fact. Facts not in ground truth = hallucination.
B. The response MUST be in English. Dutch or mixed = fp_dutch_response.
C. <think> blocks or chain-of-thought visible in response = fp_think_leaked.
D. Output SCORES first, then reasoning. Output STRICT JSON only.
E. A correct refusal when ground truth is empty scores rag_retrieved_correct=0 but NOT hallucination."""

JUDGE_PROMPT_TMPL = """CV_LABEL: {cv_label}
EXPECTED_DB_TERM: {db_search_term}
ARTIFACT_NAME: {artifact_name}

GROUND_TRUTH (from knowledge_base for this artifact):
\"\"\"{ground_truth}\"\"\"

{section_a_label}:
{section_a_content}

{section_b_label}:
{section_b_content}

EVALUATION TASK:
Assess the UGUIDU_RESPONSE against GROUND_TRUTH for the artifact identified by CV_LABEL.

SCORES (all required):
- factual_accuracy: integer 0–3
  3 = all facts fully supported by ground truth
  2 = mostly correct, minor gaps or omissions
  1 = partially correct, significant factual errors
  0 = mostly wrong, no relevant content, or describes wrong artifact
- hallucination: 1 if response states facts NOT traceable to ground truth, else 0
- rag_retrieved_correct: 1 if response is about the correct artifact, else 0
- response_in_english: 1 if fully English, else 0
- ragtruth_label: one of [correct|evident_conflict|subtle_conflict|evident_baseless|subtle_baseless]

FAILURE PATTERNS (each 0 or 1):
- fp_hallucination: facts in response not traceable to ground truth
- fp_wrong_artifact: response describes a different artifact than CV_LABEL/ARTIFACT_NAME
- fp_no_rag_hit: no RAG retrieval evident (generic answer, fallback, or I-don't-know)
- fp_dutch_response: response is in Dutch or contains significant Dutch text
- fp_label_mismatch: LABEL_TO_DB mapping produced wrong artifact (based on response content)
- fp_think_leaked: <think> tags, internal reasoning, or chain-of-thought visible in response

overall_pass: 1 if factual_accuracy >= 2 AND response_in_english == 1 AND fp_hallucination == 0, else 0

Output EXACTLY this JSON (no markdown, no extra text):
{{"scores":{{"factual_accuracy":0,"hallucination":0,"rag_retrieved_correct":0,"response_in_english":0,"ragtruth_label":"correct"}},"failure_patterns":{{"fp_hallucination":0,"fp_wrong_artifact":0,"fp_no_rag_hit":0,"fp_dutch_response":0,"fp_label_mismatch":0,"fp_think_leaked":0}},"overall_pass":0,"reasoning":"max 50 words AFTER scores"}}"""


def evaluate_once(test_case: dict, uguidu_response: str, run_idx: int) -> dict:
    """Single judge run with position randomization (bias mitigation #1 + #5)."""
    sections = [
        ('GROUND_TRUTH', test_case['ground_truth'][:3000] or '(no ground truth found in DB)'),
        ('UGUIDU_RESPONSE', uguidu_response[:2000]),
    ]
    # Seeded shuffle so runs are reproducible but positions differ across runs
    rng = random.Random(hash((test_case['id'], run_idx)))
    rng.shuffle(sections)

    prompt = JUDGE_PROMPT_TMPL.format(
        cv_label=test_case['cv_label'],
        db_search_term=test_case['db_search_term'],
        artifact_name=test_case['artifact_name'],
        ground_truth=test_case['ground_truth'][:3000] or '(no ground truth found in DB)',
        section_a_label=sections[0][0],
        section_a_content=sections[0][1],
        section_b_label=sections[1][0],
        section_b_content=sections[1][1],
    )

    # temperature=0.5 (bias mitigation #2)
    raw = gpt_chat(
        [{'role': 'system', 'content': JUDGE_SYSTEM},
         {'role': 'user', 'content': prompt}],
        temperature=0.5, max_completion_tokens=800
    )
    raw = re.sub(r'^```(?:json)?|```$', '', raw, flags=re.M).strip()
    result = json.loads(raw)

    # Store per-run binary scores for Krippendorff alpha
    s = result.get('scores', {})
    result['_binary'] = {
        'hallucination':        s.get('hallucination'),
        'rag_retrieved_correct': s.get('rag_retrieved_correct'),
        'response_in_english':  s.get('response_in_english'),
        'factual_accuracy_ge2': 1 if s.get('factual_accuracy', 0) >= 2 else 0,
        'overall_pass':         result.get('overall_pass'),
    }
    return result


def majority_vote_uguidu(runs: list) -> dict:
    """Aggregate 3 runs via majority vote (bias mitigation #4)."""
    if not runs:
        return None
    out = {}

    # factual_accuracy: median (ordinal, 0–3)
    fa_vals = [r['scores'].get('factual_accuracy', 0) for r in runs]
    out['scores'] = {'factual_accuracy': int(np.median(fa_vals))}

    # Binary scores: majority vote (≥ 2/3)
    for dim in ['hallucination', 'rag_retrieved_correct', 'response_in_english']:
        votes = [r['scores'].get(dim, 0) for r in runs]
        out['scores'][dim] = 1 if sum(votes) >= len(votes) / 2 else 0

    # ragtruth_label: plurality vote (bias mitigation #4)
    labels = [r['scores'].get('ragtruth_label', 'correct') for r in runs]
    out['scores']['ragtruth_label'] = Counter(labels).most_common(1)[0][0]

    # Failure patterns: majority vote
    fp_keys = list(runs[0].get('failure_patterns', {}).keys())
    out['failure_patterns'] = {
        k: (1 if sum(r.get('failure_patterns', {}).get(k, 0) for r in runs)
               >= len(runs) / 2 else 0)
        for k in fp_keys
    }

    # overall_pass: factual_accuracy >= 2 AND english AND NOT hallucination
    out['overall_pass'] = 1 if (
        out['scores']['factual_accuracy'] >= 2 and
        out['scores']['response_in_english'] == 1 and
        out['failure_patterns'].get('fp_hallucination', 0) == 0
    ) else 0

    # Self-consistency: fraction of binary dims where all 3 runs agreed
    binary_dims = ['hallucination', 'rag_retrieved_correct', 'response_in_english']
    n_agree = sum(
        1 for d in binary_dims
        if len(set(r['scores'].get(d) for r in runs)) == 1
    )
    out['self_consistency'] = round(n_agree / len(binary_dims), 3)

    # Per-run binary scores for Krippendorff alpha
    out['_run_binary'] = [r.get('_binary', {}) for r in runs]

    return out


def evaluate_test_case(test_case: dict, uguidu_response: str) -> dict | None:
    """Run 3 judge evaluations and aggregate via majority vote."""
    runs = []
    for i in range(JUDGE_RUNS):
        try:
            runs.append(evaluate_once(test_case, uguidu_response, i))
        except Exception as e:
            log.warning(f'Judge run {i} failed for {test_case["id"]}: {e}')
    return majority_vote_uguidu(runs) if runs else None


print(f'Evaluator ready — {JUDGE_RUNS}-run self-consistency, 6 failure patterns, 5 bias mitigations')

## Cell 8 — Main Pipeline (Resume-Safe)

Checkpoint saved after every test case. Re-run this cell to resume from where it stopped.
Delete `eval_output/eval_checkpoint.json` to start fresh.

In [ ]:
# Load checkpoint if it exists
if CKPT_PATH.exists():
    with open(CKPT_PATH, encoding='utf-8') as f:
        results = json.load(f)
    done_ids = {r['id'] for r in results}
    log.info(f'Resuming — {len(results)}/{len(test_cases)} test cases already done')
else:
    results = []
    done_ids = set()

todo = [tc for tc in test_cases if tc['id'] not in done_ids]
log.info(f'{len(todo)} test cases remaining')

for tc in tqdm(todo, desc='Evaluating artifacts'):
    record = {
        'id':              tc['id'],
        'cv_label':        tc['cv_label'],
        'db_search_term':  tc['db_search_term'],
        'artifact_id':     tc['artifact_id'],
        'artifact_name':   tc['artifact_name'],
        'ground_truth_len': len(tc['ground_truth']),
    }

    # Query UGUIDU
    try:
        uguidu_response = query_uguidu(tc['cv_label'])
        time.sleep(API_DELAY)
    except Exception as e:
        uguidu_response = f'ERROR: {e}'
        log.warning(f'API failed for {tc["id"]}: {e}')

    record['uguidu_response'] = uguidu_response

    # Run judge (skip on API error)
    if not uguidu_response.startswith('ERROR'):
        vote = evaluate_test_case(tc, uguidu_response)
        record['vote'] = vote
    else:
        record['vote'] = None

    results.append(record)

    # Checkpoint after every result
    with open(CKPT_PATH, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    # Progress log
    if record['vote']:
        fa  = record['vote']['scores'].get('factual_accuracy', '?')
        op  = record['vote'].get('overall_pass', '?')
        sc  = record['vote'].get('self_consistency', '?')
        log.info(f"{tc['id']} | {tc['cv_label'][:40]:<40} | FA={fa}/3 pass={op} sc={sc}")

log.info(f'Pipeline complete — {len(results)} results at {CKPT_PATH}')
print(f'\nDone — {len(results)} results  checkpoint: {CKPT_PATH}')

## Cell 9 — Statistical Analysis

Five statistical methods:
1. **Wilson score intervals** (Agresti & Coull 1998) — 95% CI per metric and failure pattern
2. **Krippendorff α** from individual run binary scores (nominal scale)
3. **BH-FDR correction** at q=0.05 across all 6 failure patterns (Benjamini & Hochberg 1995)
4. **Phi-coefficient** — failure pattern pair correlations
5. **Binomial tests** — pre-registered one-sided tests vs 10% baseline prevalence

In [ ]:
try:
    from statsmodels.stats.proportion import proportion_confint
    from statsmodels.stats.multitest import multipletests
    from scipy.stats import binomtest
    import krippendorff
    HAS_STATS = True
except ImportError:
    HAS_STATS = False
    print('WARNING: statsmodels / scipy / krippendorff not installed — CIs unavailable')

# Load checkpoint
with open(CKPT_PATH, encoding='utf-8') as f:
    results = json.load(f)

# ── Flatten results into DataFrame ───────────────────────────
rows = []
kripp_data = {dim: [] for dim in [
    'hallucination', 'rag_retrieved_correct', 'response_in_english',
    'factual_accuracy_ge2', 'overall_pass'
]}

for r in results:
    v = r.get('vote')
    if not v:
        continue
    s  = v.get('scores', {})
    fp = v.get('failure_patterns', {})

    row = {
        'id':                  r['id'],
        'cv_label':            r['cv_label'],
        'db_search_term':      r['db_search_term'],
        'artifact_name':       r['artifact_name'],
        'artifact_id':         r['artifact_id'],
        'ground_truth_len':    r.get('ground_truth_len', 0),
        'uguidu_response':     r.get('uguidu_response', '')[:400],
        'factual_accuracy':    s.get('factual_accuracy'),
        'hallucination':       s.get('hallucination'),
        'rag_retrieved_correct': s.get('rag_retrieved_correct'),
        'response_in_english': s.get('response_in_english'),
        'ragtruth_label':      s.get('ragtruth_label'),
        'overall_pass':        v.get('overall_pass'),
        'self_consistency':    v.get('self_consistency'),
    }
    for k, val in fp.items():
        row[k] = val
    rows.append(row)

    # Krippendorff alpha data (items × runs matrix)
    rb = v.get('_run_binary', [])
    if len(rb) == JUDGE_RUNS:
        for dim in kripp_data:
            kripp_data[dim].append([run.get(dim) for run in rb])

df = pd.DataFrame(rows)
fp_cols = [c for c in df.columns if c.startswith('fp_')]

# ── Report ────────────────────────────────────────────────────
SEP = '=' * 68
print(SEP)
print('UGUIDU RAG EVALUATION — FULL STATISTICAL REPORT')
print(f'Date:      {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'Artifacts: {len(df)} test cases | Judge: {JUDGE_MODEL} ({JUDGE_RUNS}-run majority vote)')
print(SEP)

if len(df) == 0:
    print('No data — run Cell 8 first')
else:
    # ── 1. Overall ────────────────────────────────────────────
    print('\n--- 1. OVERALL ---')
    n_pass = int(df['overall_pass'].sum())
    print(f'Overall pass rate:         {df["overall_pass"].mean()*100:5.1f}%  ({n_pass}/{len(df)})')
    print(f'Mean factual accuracy:     {df["factual_accuracy"].mean():5.2f} / 3.0')
    print(f'Mean self-consistency:     {df["self_consistency"].mean():5.3f}  (1.000 = all {JUDGE_RUNS} runs agreed)')

    # ── 2. Per-metric Wilson CI ───────────────────────────────
    print('\n--- 2. METRIC BREAKDOWN (Wilson 95% CI) ---')
    for m in ['rag_retrieved_correct', 'response_in_english', 'hallucination', 'overall_pass']:
        if m not in df.columns:
            continue
        col = df[m].dropna()
        k, n = int(col.sum()), len(col)
        rate = k / n if n > 0 else 0
        if HAS_STATS and n >= 5:
            method = 'wilson' if n >= 10 else 'beta'
            lo, hi = proportion_confint(k, n, alpha=0.05, method=method)
            ci_str = f'[{lo*100:.0f}%–{hi*100:.0f}%]'
        else:
            ci_str = 'N/A (n<5)'
        print(f'  {m:<30}  {rate*100:5.1f}%  n={k}/{n}  95% CI {ci_str}')

    print('\n  factual_accuracy distribution:')
    for score in [3, 2, 1, 0]:
        cnt = (df['factual_accuracy'] == score).sum()
        bar = '█' * cnt
        print(f'    score={score}: {cnt:2d}/{len(df)}  {bar}')

    # ── 3. Krippendorff alpha ─────────────────────────────────
    print('\n--- 3. INTER-RUN AGREEMENT (Krippendorff α) ---')
    print('Thresholds: α ≥ 0.80 publishable | 0.67–0.80 adequate | < 0.67 re-prompt')
    kripp_results = {}
    if HAS_STATS:
        for dim, item_runs in kripp_data.items():
            if not item_runs:
                continue
            # matrix shape: items × runs → transpose to runs × items
            matrix = np.array(
                [[v if v is not None else np.nan for v in run]
                 for run in item_runs],
                dtype=float
            )
            try:
                alpha = krippendorff.alpha(
                    reliability_data=matrix.T,
                    level_of_measurement='nominal'
                )
            except Exception as e:
                alpha = None
                log.warning(f'Krippendorff failed for {dim}: {e}')

            kripp_results[dim] = alpha
            marginal = np.nanmean(matrix)
            skew_note = '  ← near-uniform distribution' if (marginal > 0.90 or marginal < 0.10) else ''
            a_str = f'{alpha:.3f}' if alpha is not None else 'N/A'
            print(f'  {dim:<30}  α = {a_str}{skew_note}')
    else:
        print('  (krippendorff package not installed)')

    # ── 4. RAGTruth label distribution ───────────────────────
    print('\n--- 4. RAGTRUTH LABEL DISTRIBUTION ---')
    if 'ragtruth_label' in df.columns:
        for label, count in df['ragtruth_label'].value_counts().items():
            pct = count / len(df) * 100
            bar = '█' * int(pct / 5)
            print(f'  {label:<30}  {count:2d} ({pct:5.1f}%) {bar}')

    # ── 5. Failure pattern prevalence + BH-FDR ───────────────
    print('\n--- 5. FAILURE PATTERN PREVALENCE (Wilson CI + BH-FDR) ---')
    print(f'  H0 baseline: prevalence ≤ 10% (one-sided binomial, BH corrected)')
    fp_rows = []
    for fp in fp_cols:
        if fp not in df.columns:
            continue
        col = df[fp].dropna()
        k, n = int(col.sum()), len(col)
        if n == 0:
            continue
        prev = k / n
        lo = hi = pval = None
        if HAS_STATS and n >= 5:
            method = 'wilson' if n >= 10 else 'beta'
            lo, hi = proportion_confint(k, n, alpha=0.05, method=method)
            pval = binomtest(k, n, p=0.10, alternative='greater').pvalue
        ci_str = f'[{lo*100:.0f}%–{hi*100:.0f}%]' if lo is not None else 'N/A'
        p_str  = f'p={pval:.3f}' if pval is not None else ''
        print(f'  {fp:<35}  {prev*100:5.1f}%  {k}/{n}  {ci_str}  {p_str}')
        fp_rows.append({
            'pattern':              fp,
            'triggered':            k,
            'total':                n,
            'prevalence_pct':       round(prev * 100, 1),
            'wilson_lo_pct':        round(lo * 100, 1) if lo is not None else None,
            'wilson_hi_pct':        round(hi * 100, 1) if hi is not None else None,
            'p_vs_10pct_baseline':  round(pval, 4) if pval is not None else None,
            'krippendorff_alpha':   kripp_results.get(fp.replace('fp_', '')),
        })

    fp_df = pd.DataFrame(fp_rows)
    if HAS_STATS and len(fp_df) > 0 and 'p_vs_10pct_baseline' in fp_df.columns:
        pvals_series = fp_df['p_vs_10pct_baseline'].dropna()
        if len(pvals_series) > 0:
            _, adj, _, _ = multipletests(pvals_series.values, method='fdr_bh', alpha=0.05)
            fp_df.loc[pvals_series.index, 'bh_adj_p'] = adj.round(4)
            fp_df['sig_after_bh'] = fp_df['bh_adj_p'] < 0.05
        sig_n = fp_df.get('sig_after_bh', pd.Series(dtype=bool)).sum()
        print(f'\n  BH-FDR (q < 0.05): {sig_n}/{len(fp_df)} patterns significant')

    # ── 6. Phi-coefficient correlations ──────────────────────
    print('\n--- 6. FAILURE PATTERN CORRELATIONS (φ-coefficient) ---')
    pairs = [
        ('fp_hallucination',  'fp_no_rag_hit'),
        ('fp_hallucination',  'fp_wrong_artifact'),
        ('fp_label_mismatch', 'fp_wrong_artifact'),
        ('fp_no_rag_hit',     'fp_label_mismatch'),
    ]
    for fp1, fp2 in pairs:
        if fp1 not in df.columns or fp2 not in df.columns:
            continue
        sub = df[[fp1, fp2]].dropna()
        if len(sub) < 5:
            continue
        phi = sub[fp1].corr(sub[fp2])
        if phi is not None:
            interp = 'strong' if abs(phi) > 0.5 else ('moderate' if abs(phi) > 0.3 else 'weak')
            print(f'  {fp1} ↔ {fp2}:  φ = {phi:.3f}  ({interp})')

    # ── 7. Binomial confirmation tests ────────────────────────
    print('\n--- 7. BINOMIAL CONFIRMATION TESTS (pre-registered, one-sided) ---')
    confirmed = [
        ('fp_hallucination',  'H1: hallucination rate > 10%'),
        ('fp_no_rag_hit',     'H1: RAG miss rate > 10%'),
        ('fp_wrong_artifact', 'H1: wrong-artifact rate > 10%'),
    ]
    if HAS_STATS:
        for fp, hypothesis in confirmed:
            if fp not in df.columns:
                continue
            col = df[fp].dropna()
            k, n = int(col.sum()), len(col)
            if n == 0:
                continue
            bt = binomtest(k, n, p=0.10, alternative='greater')
            if n >= 5:
                lo, hi = proportion_confint(k, n, alpha=0.05, method='wilson' if n >= 10 else 'beta')
                ci_str = f'[{lo*100:.0f}%–{hi*100:.0f}%]'
            else:
                ci_str = 'N/A'
            sig = 'CONFIRMED p<0.05' if bt.pvalue < 0.05 else 'n.s.'
            print(f'  {fp:<35}  {k}/{n} ({k/n*100:.0f}%)  p={bt.pvalue:.4f}  {ci_str}  {sig}')
            print(f'    {hypothesis}')

    # ── 8. Worst performers ───────────────────────────────────
    print('\n--- 8. WORST PERFORMERS (by factual accuracy) ---')
    worst_cols = ['id', 'cv_label', 'artifact_name', 'factual_accuracy',
                  'ragtruth_label', 'overall_pass'] + fp_cols[:4]
    worst = df.nsmallest(min(5, len(df)), 'factual_accuracy')[worst_cols]
    print(worst.to_string(index=False))

    print(f'\n{SEP}')
    print('ANALYSIS COMPLETE')
    print(SEP)

## Cell 10 — Export Results

Writes `eval_results.csv` and `eval_summary.xlsx` with 4 sheets:
- **All Artifacts** — full per-artifact results sorted by factual accuracy
- **Failure Pattern Prevalence** — Wilson CI + BH-FDR per pattern
- **Worst Performers** — bottom 10 by factual accuracy with full detail
- **RAGTruth Labels** — distribution of RAGTruth verdicts

In [ ]:
# ── CSV ───────────────────────────────────────────────────────
csv_path = OUTPUT_DIR / 'eval_results.csv'
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'CSV exported:   {csv_path}')

# ── Excel (4 sheets) ─────────────────────────────────────────
xlsx_path = OUTPUT_DIR / 'eval_summary.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:

    # Sheet 1: All Artifacts
    df.sort_values('factual_accuracy').to_excel(
        writer, sheet_name='All Artifacts', index=False
    )

    # Sheet 2: Failure Pattern Prevalence
    if 'fp_df' in dir() and len(fp_df) > 0:
        fp_df_export = fp_df.copy()
        fp_df_export.to_excel(
            writer, sheet_name='Failure Pattern Prevalence', index=False
        )
    else:
        # Rebuild fp_df if Cell 9 was not run in the same session
        fp_rows_export = []
        for fp in fp_cols:
            if fp not in df.columns:
                continue
            col = df[fp].dropna()
            k, n = int(col.sum()), len(col)
            if n == 0:
                continue
            fp_rows_export.append({
                'pattern': fp,
                'triggered': k,
                'total': n,
                'prevalence_pct': round(k / n * 100, 1),
            })
        pd.DataFrame(fp_rows_export).to_excel(
            writer, sheet_name='Failure Pattern Prevalence', index=False
        )

    # Sheet 3: Worst Performers
    worst_detail_cols = [
        'id', 'cv_label', 'artifact_name', 'artifact_id',
        'factual_accuracy', 'hallucination', 'rag_retrieved_correct',
        'response_in_english', 'ragtruth_label', 'overall_pass',
        'self_consistency', 'uguidu_response'
    ] + fp_cols
    worst_detail_cols = [c for c in worst_detail_cols if c in df.columns]
    df.nsmallest(min(10, len(df)), 'factual_accuracy')[worst_detail_cols].to_excel(
        writer, sheet_name='Worst Performers', index=False
    )

    # Sheet 4: RAGTruth Labels
    if 'ragtruth_label' in df.columns:
        rt = df['ragtruth_label'].value_counts().reset_index()
        rt.columns = ['label', 'count']
        rt['pct'] = (rt['count'] / rt['count'].sum() * 100).round(1)
        rt.to_excel(writer, sheet_name='RAGTruth Labels', index=False)

print(f'Excel exported: {xlsx_path}')
print('Sheets: All Artifacts | Failure Pattern Prevalence | Worst Performers | RAGTruth Labels')
print(f'\nEvaluation complete — {len(df)} artifacts assessed')